# V20: Daten verknüpfen — Datenbank-Detektiv 🔗

Heute verknüpfen wir Tabellen und lösen Fälle als Datenbank-Detektiv!

| Abschnitt | Punkte |
|-----------|--------|
| Warm-Up | 1 |
| Guided: JOIN | 2 |
| Challenge: Detektiv | 3 |
| Bonus: Wartungs-Tabelle | 5 |
| **Gesamt** | **max. 11** |

---
## Warm-Up: SQL-Wiederholung (1 Punkt)

Wiederhole die Basics aus V19.

In [ ]:
import sqlite3

verbindung = sqlite3.connect("warmup.db")
cursor = verbindung.cursor()

# Tabelle erstellen und befüllen
cursor.execute("CREATE TABLE IF NOT EXISTS tiere (id INTEGER PRIMARY KEY, name TEXT, art TEXT, alter_jahre INTEGER)")
cursor.execute("DELETE FROM tiere")  # Für Neustart
cursor.execute("INSERT INTO tiere VALUES (1, 'Bello', 'Hund', 5)")
cursor.execute("INSERT INTO tiere VALUES (2, 'Minka', 'Katze', 3)")
cursor.execute("INSERT INTO tiere VALUES (3, 'Rex', 'Hund', 8)")
cursor.execute("INSERT INTO tiere VALUES (4, 'Felix', 'Katze', 2)")
verbindung.commit()

# TODO: Zeige alle Tiere
cursor.execute(___)
print("Alle Tiere:")
for z in cursor.fetchall():
    print(f"  {z}")

# TODO: Zeige nur Hunde (WHERE art = 'Hund')
cursor.execute(___)
print("\nNur Hunde:")
for z in cursor.fetchall():
    print(f"  {z}")

verbindung.close()

---
## Theorie: Mehrere Tabellen + JOIN

### Warum zwei Tabellen?

**Eine Tabelle** → Daten werden doppelt gespeichert ("Halle A" steht 5x drin)

**Zwei Tabellen** → Jede Info steht nur einmal, verknüpft über **Fremdschlüssel**

### JOIN verknüpft Tabellen

```sql
SELECT maschinen.name, hallen.name
FROM maschinen
JOIN hallen ON maschinen.hallen_id = hallen.id
```

Lies es als: *"Nimm jede Maschine und schlag nach, zu welcher Halle sie gehört."*

### GROUP BY + COUNT: Zusammenfassen

```sql
SELECT hallen.name, COUNT(*) as anzahl
FROM maschinen
JOIN hallen ON maschinen.hallen_id = hallen.id
GROUP BY hallen.name
```

→ "Wie viele Maschinen pro Halle?"

---
## Demo: Datenbank mit zwei Tabellen

In [ ]:
import sqlite3

verbindung = sqlite3.connect("fabrik_demo.db")
cursor = verbindung.cursor()

# Zwei Tabellen erstellen
cursor.execute("DROP TABLE IF EXISTS hallen")
cursor.execute("DROP TABLE IF EXISTS maschinen")

cursor.execute("""
    CREATE TABLE hallen (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        adresse TEXT
    )
""")

cursor.execute("""
    CREATE TABLE maschinen (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        typ TEXT,
        temperatur REAL,
        hallen_id INTEGER
    )
""")

# Hallen
cursor.execute("INSERT INTO hallen (name, adresse) VALUES ('Halle A', 'Industriestr. 1')")
cursor.execute("INSERT INTO hallen (name, adresse) VALUES ('Halle B', 'Industriestr. 3')")
cursor.execute("INSERT INTO hallen (name, adresse) VALUES ('Halle C', 'Fabrikweg 5')")

# Maschinen (hallen_id verweist auf hallen.id)
cursor.execute("INSERT INTO maschinen (name, typ, temperatur, hallen_id) VALUES ('CNC-01', 'Fräse', 42.5, 1)")
cursor.execute("INSERT INTO maschinen (name, typ, temperatur, hallen_id) VALUES ('CNC-02', 'Fräse', 38.1, 1)")
cursor.execute("INSERT INTO maschinen (name, typ, temperatur, hallen_id) VALUES ('CNC-03', 'Fräse', 51.0, 2)")
cursor.execute("INSERT INTO maschinen (name, typ, temperatur, hallen_id) VALUES ('ROBO-01', 'Roboter', 55.0, 2)")
cursor.execute("INSERT INTO maschinen (name, typ, temperatur, hallen_id) VALUES ('ROBO-02', 'Roboter', 33.0, 3)")
cursor.execute("INSERT INTO maschinen (name, typ, temperatur, hallen_id) VALUES ('LASER-01', 'Laser', 28.0, 3)")
cursor.execute("INSERT INTO maschinen (name, typ, temperatur, hallen_id) VALUES ('LASER-02', 'Laser', 62.0, 1)")

verbindung.commit()
print("Datenbank mit 3 Hallen und 7 Maschinen erstellt!")

In [ ]:
# JOIN: Maschinen + Hallen zusammen anzeigen
cursor.execute("""
    SELECT maschinen.name, maschinen.typ, maschinen.temperatur, hallen.name
    FROM maschinen
    JOIN hallen ON maschinen.hallen_id = hallen.id
""")

print("Alle Maschinen mit Standort:")
for z in cursor.fetchall():
    print(f"  {z[0]:10s} | {z[1]:8s} | {z[2]:5.1f}°C | {z[3]}")

In [ ]:
# GROUP BY: Maschinen pro Halle zählen
cursor.execute("""
    SELECT hallen.name, COUNT(*) as anzahl
    FROM maschinen
    JOIN hallen ON maschinen.hallen_id = hallen.id
    GROUP BY hallen.name
""")

print("Maschinen pro Halle:")
for z in cursor.fetchall():
    print(f"  {z[0]}: {z[1]} Maschinen")

---
## Guided Exercise: JOIN üben (2 Punkte)

Benutze die Datenbank von oben und schreibe eigene Abfragen.

In [ ]:
# Aufgabe A: Zeige nur Maschinen in "Halle A"
# Tipp: JOIN + WHERE hallen.name = 'Halle A'
cursor.execute("""
    SELECT maschinen.name, maschinen.temperatur
    FROM maschinen
    JOIN hallen ON maschinen.hallen_id = hallen.id
    WHERE ___
""")

print("Maschinen in Halle A:")
for z in cursor.fetchall():
    print(f"  {z[0]}: {z[1]}°C")

In [ ]:
# Aufgabe B: Durchschnittstemperatur pro Halle
# Tipp: AVG(maschinen.temperatur) + GROUP BY
cursor.execute("""
    SELECT hallen.name, ___ as durchschnitt
    FROM maschinen
    JOIN hallen ON maschinen.hallen_id = hallen.id
    GROUP BY ___
""")

print("Durchschnittstemperatur pro Halle:")
for z in cursor.fetchall():
    print(f"  {z[0]}: {z[1]:.1f}°C")

---
## Challenge: Datenbank-Detektiv (3 Punkte)

Löse die folgenden Fälle nur mit SQL-Abfragen!

In [ ]:
# Fall 1: Welche Maschine ist am heißesten und wo steht sie?
# Tipp: ORDER BY temperatur DESC LIMIT 1
cursor.execute("""
    SELECT maschinen.name, maschinen.temperatur, hallen.name
    FROM maschinen
    JOIN hallen ON maschinen.hallen_id = hallen.id
    ORDER BY ___
    LIMIT ___
""")

z = cursor.fetchone()
print(f"Fall 1: {z[0]} mit {z[1]}°C in {z[2]}")

In [ ]:
# Fall 2: Welche Halle hat die meisten Maschinen?
# Tipp: GROUP BY + COUNT + ORDER BY anzahl DESC LIMIT 1
cursor.execute("""
    SELECT hallen.name, COUNT(*) as anzahl
    FROM maschinen
    JOIN hallen ON maschinen.hallen_id = hallen.id
    GROUP BY hallen.name
    ORDER BY ___
    LIMIT ___
""")

z = cursor.fetchone()
print(f"Fall 2: {z[0]} mit {z[1]} Maschinen")

In [ ]:
# Fall 3: Welche Maschinentypen gibt es und wie viele jeweils?
# Tipp: GROUP BY typ
cursor.execute("""
    SELECT typ, COUNT(*) as anzahl
    FROM maschinen
    GROUP BY ___
""")

print("Fall 3 — Maschinentypen:")
for z in cursor.fetchall():
    print(f"  {z[0]}: {z[1]}x")

---
## Bonus: Wartungs-Protokoll (5 Punkte)

Erstelle eine dritte Tabelle `wartungen` und verknüpfe sie.

In [ ]:
# Wartungs-Tabelle erstellen
cursor.execute("DROP TABLE IF EXISTS wartungen")
cursor.execute("""
    CREATE TABLE wartungen (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        maschinen_id INTEGER,
        datum TEXT,
        beschreibung TEXT
    )
""")

# TODO: Füge mindestens 4 Wartungseinträge ein
cursor.execute("INSERT INTO wartungen (maschinen_id, datum, beschreibung) VALUES (1, '2024-01-15', 'Ölwechsel')")
cursor.execute("INSERT INTO wartungen (maschinen_id, datum, beschreibung) VALUES (1, '2024-06-20', 'Lager getauscht')")
cursor.execute("INSERT INTO wartungen (maschinen_id, datum, beschreibung) VALUES (4, '2024-03-10', 'Software-Update')")
cursor.execute("INSERT INTO wartungen (maschinen_id, datum, beschreibung) VALUES (7, '2024-05-01', 'Linse gereinigt')")
verbindung.commit()

# TODO: JOIN über maschinen und wartungen
# Zeige: Maschinenname, Datum, Beschreibung
cursor.execute("""
    SELECT maschinen.name, wartungen.datum, wartungen.beschreibung
    FROM wartungen
    JOIN maschinen ON wartungen.maschinen_id = maschinen.id
    ORDER BY wartungen.datum
""")

print("Wartungsprotokoll:")
for z in cursor.fetchall():
    print(f"  {z[1]} | {z[0]}: {z[2]}")

# TODO: Welche Maschine wurde am häufigsten gewartet?
cursor.execute("""
    SELECT maschinen.name, COUNT(*) as anzahl
    FROM wartungen
    JOIN maschinen ON wartungen.maschinen_id = maschinen.id
    GROUP BY maschinen.name
    ORDER BY anzahl DESC
""")

print("\nWartungen pro Maschine:")
for z in cursor.fetchall():
    print(f"  {z[0]}: {z[1]}x gewartet")

verbindung.close()

---
## Zusammenfassung — GESAMTER KURS

| Vorlesung | Thema | Python Highlight |
|-----------|-------|------------------|
| V11 | KI und LLMs | print, input, if |
| V12 | Prompt Engineering | f-string, .strip(), .upper() |
| V13 | Rechner-Architektur | matplotlib.pyplot.bar() |
| V14 | Betriebssysteme | os.listdir(), os.path |
| V15 | Netzwerk-Grundlagen | def, return (Funktionen) |
| V16 | APIs und JSON | requests.get(), .json() |
| V17 | Caesar-Chiffre | ord(), chr() |
| V18 | Passwörter/Hashes | hashlib.md5() |
| V19 | Datenbank-Basics | sqlite3, SELECT/INSERT |
| V20 | Daten verknüpfen | JOIN, GROUP BY, COUNT |

**Viel Erfolg bei der Klausur!** 🎓